# Torsion Test — Modulus of Rigidity (G)

### Student-friendly analysis tool
Enter your laboratory readings in the table below, choose how many initial readings you want to test as the elastic/proportional region, and click **Analyze**.

The notebook calculates the linear-regression slope and modulus of rigidity using
\[
\frac{T}{J}=\frac{G\theta}{L},\qquad G=\frac{mL}{J}
\]
with \(\theta\) in radians and torque converted to N·mm.

**Important:** The default elastic-point count is only a starting value. Students should try a few more or fewer initial readings and inspect the fitted line, \(R^2\), and calculated \(G\) before deciding which points represent the linear/proportional region.


## 1. Specimen information

These defaults correspond to the mild-steel specimen discussed for this experiment. Change them if your specimen dimensions are different.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# Specimen dimensions
diameter = widgets.FloatText(value=8.1, description='Diameter d (mm):', layout=widgets.Layout(width='260px'))
gauge_length = widgets.FloatText(value=91.0, description='Gauge length L (mm):', layout=widgets.Layout(width='280px'))

# Torque unit recorded by the machine
torque_unit = widgets.Dropdown(
    options=[('kgf·cm', 'kgf-cm'), ('kgf·m', 'kgf-m')],
    value='kgf-cm', description='Torque unit:',
    layout=widgets.Layout(width='250px')
)

# Number of initial points to use in the regression
elastic_count = widgets.IntSlider(
    value=7, min=2, max=30, step=1,
    description='Elastic points:',
    continuous_update=False,
    style={'description_width':'initial'},
    layout=widgets.Layout(width='400px')
)

display(widgets.HBox([diameter, gauge_length]))
display(widgets.HBox([torque_unit, elastic_count]))


## 2. Enter the readings

Enter one reading per row. The angle is in **degrees**. Torque is in the unit selected above.

- Rows are pre-filled with angles 1–20° to make the common small-angle elastic readings quick to enter.
- Leave unused rows blank.
- Use **Add 10 rows** if the test goes beyond the displayed rows.
- The first `Elastic points` rows are used for the regression.
- You can change the elastic-point slider and click **Analyze** again without re-entering the readings.


In [ ]:
rows = []
table_box = widgets.VBox()

def make_row(i, preset_angle=None):
    angle = widgets.FloatText(
        value=preset_angle if preset_angle is not None else 0,
        description='', layout=widgets.Layout(width='145px')
    )
    torque = widgets.FloatText(
        value=0, description='', layout=widgets.Layout(width='170px')
    )
    angle._row_index = i
    torque._row_index = i
    return widgets.HBox([
        widgets.Label(str(i+1), layout=widgets.Layout(width='35px')),
        angle,
        torque
    ])

def rebuild_table(n_rows):
    global rows
    rows = []
    children = [widgets.HBox([
        widgets.Label('#', layout=widgets.Layout(width='35px')),
        widgets.Label('Angle of twist θ (degrees)', layout=widgets.Layout(width='145px')),
        widgets.Label('Torque T (selected unit)', layout=widgets.Layout(width='170px'))
    ])]
    for i in range(n_rows):
        preset = i if i < 20 else None
        row = make_row(i, preset)
        rows.append(row)
        children.append(row)
    table_box.children = tuple(children)

rebuild_table(30)

add_rows_button = widgets.Button(description='Add 10 rows', button_style='')
clear_button = widgets.Button(description='Clear all readings', button_style='warning')

def add_rows(_):
    # Preserve all existing values before rebuilding the table.
    old_values = [(row.children[1].value, row.children[2].value) for row in rows]
    new_n = len(old_values) + 10
    new_rows = []
    children = [table_box.children[0]]

    for i in range(new_n):
        row = make_row(i, None)
        if i < len(old_values):
            row.children[1].value = old_values[i][0]
            row.children[2].value = old_values[i][1]
        new_rows.append(row)
        children.append(row)

    rows[:] = new_rows
    table_box.children = tuple(children)

def clear_readings(_):
    for row in rows:
        row.children[1].value = 0
        row.children[2].value = 0

add_rows_button.on_click(add_rows)
clear_button.on_click(clear_readings)
display(table_box)
display(widgets.HBox([add_rows_button, clear_button]))


## 3. Analyze the readings

Click the button after entering the readings. You can change **Elastic points** and analyze again to compare different choices.


In [ ]:
analyze_button = widgets.Button(description='Analyze readings', button_style='primary', icon='line-chart')
results = widgets.Output()
display(analyze_button, results)

def analyze(_):
    with results:
        clear_output(wait=True)
        try:
            d = float(diameter.value)
            L = float(gauge_length.value)
            nfit = int(elastic_count.value)
            if d <= 0 or L <= 0:
                raise ValueError('Diameter and gauge length must be positive.')

            readings = []
            for i, row in enumerate(rows):
                a = float(row.children[1].value)
                t = float(row.children[2].value)
                # A row is considered entered when both fields are non-zero.
                if a != 0 or t != 0:
                    if a == 0 or t == 0:
                        raise ValueError(f'Row {i+1}: enter both angle and torque, or leave the row unused.')
                    readings.append((a, t))

            if len(readings) < 3:
                raise ValueError('Enter at least 3 complete readings.')
            if nfit > len(readings):
                raise ValueError(f'Elastic points is set to {nfit}, but only {len(readings)} complete readings were entered.')

            angles_deg = np.array([r[0] for r in readings], dtype=float)
            torque_input = np.array([r[1] for r in readings], dtype=float)
            if np.any(~np.isfinite(angles_deg)) or np.any(~np.isfinite(torque_input)):
                raise ValueError('Readings contain an invalid number.')
            if np.any(np.diff(angles_deg) < 0):
                raise ValueError('Enter readings in increasing angle order.')

            if torque_unit.value == 'kgf-cm':
                factor = 98.0665
                unit_label = 'kgf·cm'
            else:
                factor = 9806.65
                unit_label = 'kgf·m'

            theta_rad = np.deg2rad(angles_deg)
            torque_Nmm = torque_input * factor
            J = np.pi * d**4 / 32

            x = theta_rad[:nfit]
            y = torque_Nmm[:nfit]
            slope, intercept = np.polyfit(x, y, 1)
            pred = slope*x + intercept
            ss_res = np.sum((y-pred)**2)
            ss_tot = np.sum((y-y.mean())**2)
            r2 = 1 - ss_res/ss_tot if ss_tot != 0 else np.nan
            G_MPa = slope*L/J
            G_GPa = G_MPa/1000
            slope_input_rad = slope/factor
            slope_input_deg = slope_input_rad*np.pi/180

            print('RESULTS')
            print('=======')
            print(f'Number of readings: {len(readings)}')
            print(f'Points used for elastic fit: {nfit}')
            print(f'J = {J:.3f} mm⁴')
            print(f'Linear fit: T = {intercept:.3f} + ({slope:.3f}) θ  [N·mm, rad]')
            print(f'Slope m = {slope:.3f} N·mm/rad')
            print(f'Slope = {slope_input_rad:.3f} {unit_label}/rad')
            print(f'Slope = {slope_input_deg:.3f} {unit_label}/degree')
            print(f'R² = {r2:.5f}')
            print(f'Modulus of rigidity G = {G_MPa:.2f} MPa = {G_GPa:.3f} GPa')

            if r2 < 0.99:
                print('\nCAUTION: R² is below 0.99. Try a different number of elastic points.')
            else:
                print('\nReminder: a high R² does not by itself prove that all selected points are physically elastic. Inspect the plot.')

            # Elastic plot
            fig, ax = plt.subplots(figsize=(9, 5.8))
            ax.scatter(angles_deg[:nfit], torque_input[:nfit], s=55, label=f'Points used for G (n={nfit})')
            xx = np.linspace(x.min(), x.max(), 100)
            yy = slope*xx + intercept
            ax.plot(np.rad2deg(xx), yy/factor, linewidth=2, label=f'Linear fit, R²={r2:.5f}')
            ax.set_xlabel('Angle of twist, θ (degrees)')
            ax.set_ylabel(f'Torque T ({unit_label})')
            ax.set_title(f'Elastic / Proportional Region — G = {G_GPa:.3f} GPa')
            ax.grid(True, alpha=0.3)
            ax.legend()
            fig.tight_layout()
            fig.savefig('torsion_elastic_region.png', dpi=600, bbox_inches='tight')
            plt.show()

            # Full curve
            fig, ax = plt.subplots(figsize=(10, 6.2))
            ax.plot(angles_deg, torque_input, marker='o', markersize=5, linewidth=1.5, label='All measured readings')
            ax.scatter(angles_deg[:nfit], torque_input[:nfit], s=65, label=f'Points used for G (n={nfit})', zorder=3)
            ax.set_xlabel('Angle of twist, θ (degrees)')
            ax.set_ylabel(f'Torque T ({unit_label})')
            ax.set_title('Complete Torque–Twist Curve')
            ax.grid(True, alpha=0.3)
            ax.legend()
            fig.tight_layout()
            fig.savefig('torsion_full_torque_twist_curve.png', dpi=600, bbox_inches='tight')
            plt.show()

            data = pd.DataFrame({
                'Angle (deg)': angles_deg,
                'Angle (rad)': theta_rad,
                f'Torque ({unit_label})': torque_input,
                'Torque (N·mm)': torque_Nmm,
                'Used for elastic fit': ['Yes' if i < nfit else 'No' for i in range(len(angles_deg))]
            })
            data.to_csv('torsion_processed_data.csv', index=False)
            display(data.round(6))
            print('\nSaved: torsion_elastic_region.png, torsion_full_torque_twist_curve.png, torsion_processed_data.csv')
        except Exception as e:
            print('INPUT ERROR:', e)

analyze_button.on_click(analyze)


## 4. Download your results

Run the following cell after analysis to download the two 600-DPI graphs and the processed CSV file.


In [ ]:
from google.colab import files
files.download('torsion_elastic_region.png')
files.download('torsion_full_torque_twist_curve.png')
files.download('torsion_processed_data.csv')


## Student checklist

- Enter all measured readings in increasing angle order.
- Try several values of **Elastic points** (for example 5, 6, 7, 8, 9) and compare the regression and calculated \(G\).
- Use the graph to decide whether the selected points are genuinely in the initial linear/proportional region.
- Use radians for \(\theta\) when calculating \(G\).
- For a solid circular specimen, \(J=\pi d^4/32\).
- Conversion: 1 kgf·cm = 98.0665 N·mm; 1 kgf·m = 9806.65 N·mm.
- The full curve is for observing the complete torque–twist behaviour; only the selected initial points are used to determine \(G\).
